In [ ]:
# from ipywidgets import interact, interactive, widgets, Layout
# from IPython.display import display, clear_output
# from pathlib import Path
# from etils import epath
# import jax

# maindir = Path('/mnt/c/Users/dpoteryayev/1. PhD/SpectraFormer/').resolve()

# datadir = maindir / "data"
# parsed_datadir = datadir / "parsed_data"

# import xarray as xr
# import matplotlib.pyplot as plt

# import numpy as np

# # Get available materials (subdirectories in parsed_datadir)
# available_materials = [d.name for d in parsed_datadir.iterdir() if d.is_dir()]

# # Create widgets
# materials_dropdown = widgets.Dropdown(
#     options=available_materials,
#     description='Material:',
#     layout=Layout(width='80%')
#     )

# dataset_dropdown = widgets.Dropdown(
#     description='Dataset:',
#     layout=Layout(width='80%'))

# save_button = widgets.Button(description="Save Current Plot")

# out = widgets.Output()

# # Function to update dataset options based on selected material
# def update_datasets(change):
#     material_path = parsed_datadir / change['new']
#     nc_files = list(material_path.glob('**/*.nc'))
#     materials_dropdown.options = [f.stem for f in nc_files]
    
# materials_dropdown.observe(update_datasets, names='value')

# # Initialize datasets for first material
# if available_materials:
#     materials_dropdown.value = available_materials[0]
#     update_datasets({'new': materials_dropdown.value})

# def load_dataset(b):
#     global ds_to_plot, material_tag
#     material_tag = materials_dropdown.value
#     nc_file = (parsed_datadir / material_tag / f"{materials_dropdown.value}.nc")
#     ds_to_plot = xr.load_dataset(nc_file)
#     sample_slider.max = len(ds_to_plot['sample'])-1
#     print(f"Loaded dataset: {material_tag}/{dataset_dropdown.value}")
#     update_plot(0)








# # Display interface
# display(widgets.VBox([
#     widgets.HBox([materials_dropdown, dataset_dropdown, 
#                 #   load_button
#                   ]),
#     # widgets.HBox([sample_slider, save_button]),
#     out
# ]))

**Raw datasets visualization**

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
from ipywidgets import widgets, Layout
from IPython.display import display, clear_output
from pathlib import Path
import matplotlib.gridspec as gridspec
import numpy as np
import sys
from scipy.signal import savgol_filter

# Configure paths
current_dir = Path.cwd()
project_root = current_dir.parent 
sys.path.insert(0, str(project_root))

maindir = Path('/mnt/c/Users/dpoteryayev/1. PhD/SpectraFormer/').resolve()
parsed_datadir = maindir / "data" / "parsed_data"

# Import preprocessing function AFTER path configuration
from spectraformer.input_pipeline import preprocess_dataset

# Configure plot layout only
plt.rcParams.update({
    'figure.constrained_layout.use': True
})

# Widgets
material_dropdown = widgets.Dropdown(description='Material:', layout={'width': '500px'})
file_dropdown = widgets.Dropdown(description='File:', layout={'width': '500px'})
filter_controls = widgets.HBox([
    widgets.IntSlider(
        value=11, 
        min=3, 
        max=51, 
        step=2,
        description='Window:',
        layout=Layout(width='33%')
    ),
    widgets.IntSlider(
        value=3, 
        min=1, 
        max=7, 
        description='Poly:',
        layout=Layout(width='33%')
    ),
    widgets.IntSlider(
        value=1, 
        min=1, 
        max=1000, 
        description='# of filtering times:',
        layout=Layout(width='33%')
    )
])
preprocess_checkbox = widgets.Checkbox(
    value=False,
    description='Preprocess data',
    disabled=False,
    indent=True
)
output = widgets.Output()

# Find materials
def find_materials():
    return sorted([d.name for d in parsed_datadir.iterdir() if d.is_dir()])

# Find NC files
def find_nc_files(material):
    return sorted((parsed_datadir / material).rglob('*.nc'))

# Initialize widgets
def init_widgets():
    materials = find_materials()
    material_dropdown.options = materials
    if materials:
        material_dropdown.value = materials[0]
        update_files()

# Update files dropdown
def update_files(*args):
    material = material_dropdown.value
    if material:
        nc_files = find_nc_files(material)
        rel_paths = [f.relative_to(parsed_datadir / material) for f in nc_files]
        file_dropdown.options = [str(p) for p in rel_paths]  # Store as strings
        if rel_paths:
            file_dropdown.value = str(rel_paths[0])
            update_plots()
        else:
            file_dropdown.value = None
            with output:
                clear_output(wait=True)
                print("No .nc files found for selected material")

material_dropdown.observe(update_files, 'value')
file_dropdown.observe(lambda _: update_plots(), 'value')
preprocess_checkbox.observe(lambda _: update_plots(), 'value')

# Statistics function
def my_statistics(data_array):
    spatial_dims = [d for d in data_array.dims if d.startswith('X_')]
    if 'spectra' not in data_array.dims:
        data_array = data_array.stack(spectra=spatial_dims)
    
    exp_val = data_array.mean(dim='spectra')
    variance = data_array.var(dim='spectra')
    std = data_array.std(dim='spectra')
    z1 = (data_array - exp_val)/std
    skewness = (z1**3).mean(dim='spectra')
    kurtosis = (z1**4).mean(dim='spectra')
    
    scale_param = variance / exp_val
    shape_param = (exp_val**2) / (std**2)
    
    return {
        'exp_val': exp_val,
        'variance': variance,
        'std': std,
        'skewness': skewness,
        'excess_kurtosis': kurtosis - 3,
        'scale_param': scale_param,
        'shape_param': shape_param
    }

# Plotting function with x-axis numbers on all subplots
def update_plots(*args):
    with output:
        clear_output(wait=True)
        plt.close('all')
        
        # Validate selection state
        if not (material_dropdown.value and file_dropdown.value):
            print("Select material and file first")
            return
        
        # Load data
        material = material_dropdown.value
        file_path = Path(file_dropdown.value)
        full_path = parsed_datadir / material / file_path
        
        # Check file existence
        if not full_path.exists():
            print(f"File not found: {full_path}")
            return
        
        window = filter_controls.children[0].value
        poly = filter_controls.children[1].value
        filtering_times = filter_controls.children[2].value
        
        ds = xr.open_dataset(full_path)
        data_array = next(iter(ds.data_vars.values()))
        
        if preprocess_checkbox.value:
            data_array = preprocess_dataset(data_array, sup_norm_threshold=0.15, option='proper_bg_proper_norm_with_outliers')
        
        if len(data_array.dims) == 1:
            data_array = data_array.expand_dims("X_0")
        
        # Prepare data
        wave_number = ds.wave_number.values
        spatial_dims = [d for d in data_array.dims if d.startswith('X_')]
        
        
        
        # Stack dimensions
        stacked = data_array.stack(spectra=spatial_dims)
        spectra = stacked.values
        mean_spectrum = spectra.mean(axis=1)
        median_spectrum = np.median(spectra, axis=1)
        
        fig3 = plt.figure(num='Filtered Spectra', figsize=(20, 10))
        ax3 = fig3.add_subplot(111)
        # Apply Savitzky-Golay filter
        filtered_spectra = spectra
        for _ in range(filtering_times):
            filtered_spectra = savgol_filter(filtered_spectra, window_length=window, polyorder=poly, axis=0)
        print(filtered_spectra.shape)
        
        filtered_data_array = xr.DataArray(
            data=filtered_spectra,
            dims=stacked.dims,
            coords=stacked.coords,
            name=stacked.name + "_filtered"
        )
        
        
        for i in range(filtered_spectra.shape[1]):
            ax3.plot(
                wave_number, 
                filtered_spectra[:, i], 
                alpha=0.8, 
                linewidth=0.8
            )
        ax3.plot(wave_number, filtered_spectra.mean(axis=1), color='red', linewidth=3, label='Mean Filtered Spectrum')
        ax3.set_title(f'Filtered Spectra: {material} / {file_path.name}', fontsize='x-large', pad=20)
        ax3.set_xlabel('Wave Number', fontsize='x-large', labelpad=15)
        ax3.set_ylabel('Intensity', fontsize='x-large', labelpad=15)
        ax3.tick_params(axis='both', which='major', labelsize='x-large')
        ax3.legend(prop={'size': 'x-large'})
        ax3.grid(True, which='both', linestyle='--', linewidth=0.5)
        
        # Create and plot Spectra figure
        fig1 = plt.figure(num='Spectra', figsize=(20, 10))
        ax1 = fig1.add_subplot(111)
        
        # colors = plt.cm.vanimo(np.linspace(0, 1, spectra.shape[1]))
        for i in range(spectra.shape[1]):
            ax1.plot(
                wave_number, 
                spectra[:, i], 
                #  color=colors[i], 
                alpha=0.8, 
                linewidth=0.8
                )
        
        ax1.plot(wave_number, mean_spectrum, color='red', linewidth=3, label='Mean Spectrum')
        ax1.plot(wave_number, median_spectrum, color='blue', linewidth=3, label='Median Spectrum')
        # ax1.plot(wave_number, median_spectrum*1.15, color='blue', linewidth=3, linestyle='--', label='Median Spectrum + 15%')
        # ax1.plot(wave_number, median_spectrum*0.85, color='blue', linewidth=3, linestyle='--', label='Median Spectrum - 15%')
        ax1.set_title(f'Spectra: {material} / {file_path.name}', fontsize='x-large', pad=20)
        ax1.set_xlabel('Wave Number', fontsize='x-large', labelpad=15)
        ax1.set_ylabel('Intensity', fontsize='x-large', labelpad=15)
        ax1.tick_params(axis='both', which='major', labelsize='x-large')
        ax1.legend(prop={'size': 'x-large'})
        ax1.grid(True, which='both', linestyle='--', linewidth=0.5)
        
        # ax1.set_ylim(-100, 500)
        
        # Create and plot Statistics figure
        stats_unfiltered = my_statistics(data_array)
        stats_filtered = my_statistics(filtered_data_array)
        
        # Figure for statistics
        fig2 = plt.figure(num='Statistics', figsize=(10, 20))
        gs = gridspec.GridSpec(3, 2, figure=fig2, hspace=0.05, wspace=0.05)
        
        plots = [
            ('variance', 'Variance'),
            ('std', 'Standard Deviation'),
            ('skewness', 'Skewness'),
            ('excess_kurtosis', 'Excess Kurtosis'),
            ('scale_param', 'Scale Parameter'),
            ('shape_param', 'Shape Parameter')
        ]
        
        for idx, (var, title) in enumerate(plots):
            ax = fig2.add_subplot(gs[idx])
            for stats in [stats_unfiltered, stats_filtered]:
                
                x_vals = stats['exp_val'].values
                y_vals = stats[var].values
                ax.scatter(x_vals, y_vals, alpha=0.6, s=50
                        #    , color='darkblue'
                           )
                ax.set_title(title, fontsize='x-large', pad=10)
                ax.set_xlabel('Mean Value', fontsize='x-large', labelpad=8)
                ax.set_ylabel(title, fontsize='x-large', labelpad=8)
                ax.tick_params(axis='both', labelsize='x-large')
                ax.grid(True, which='both', linestyle='--', linewidth=0.5)
                ax.legend(
                    ['Unfiltered', 'Filtered'], 
                    loc='upper right', 
                    prop={'size': 'x-large'}
                )
                
                # # Modified section: Show x-axis numbers on all subplots
                # if idx % 2 != 0:
                #     ax.yaxis.set_label_position("right")
                #     ax.yaxis.tick_right()
                
                # Always show x-axis numbers (removed the ax.xaxis.set_ticklabels([]) call)
        

        
        plt.show()

# Connect filter controls
for control in filter_controls.children:
    control.observe(lambda _: update_plots(), 'value')
# widgets.interactive_output(update_plots, {'material': material_dropdown, 'file_path': file_dropdown})

init_widgets()
# Display interface
display(widgets.VBox([
    widgets.HBox([material_dropdown, file_dropdown, preprocess_checkbox]),
    filter_controls,
    output
    ]))


**Preprocessed datasets visualization**

In [ ]:
# import xarray as xr
# import matplotlib.pyplot as plt
# import ipywidgets as widgets
# from IPython.display import display, clear_output
# from pathlib import Path
# import matplotlib.gridspec as gridspec
# import numpy as np
# import sys
# from scipy.signal import savgol_filter

# # Configure paths
# current_dir = Path.cwd()
# project_root = current_dir.parent 
# sys.path.insert(0, str(project_root))

# maindir = Path('/mnt/c/Users/dpoteryayev/1. PhD/SpectraFormer/').resolve()
# parsed_datadir = maindir / "data" / "parsed_data"

# # Import preprocessing function AFTER path configuration
# from spectraformer.input_pipeline import preprocess_dataset

# # Configure plot layout
# plt.rcParams.update({'figure.constrained_layout.use': True})

# # Widget setup
# material_dropdown = widgets.Dropdown(description='Material:', layout={'width': '500px'})
# file_dropdown = widgets.Dropdown(description='File:', layout={'width': '500px'})
# output = widgets.Output()

# # Modified materials discovery with debug output
# def find_materials():
#     materials_dir = parsed_datadir
#     print(f"Checking for materials in: {materials_dir}")  # Debug output
    
#     if not materials_dir.exists():
#         raise FileNotFoundError(f"Directory not found: {materials_dir}")
    
#     materials = sorted([
#         d.name for d in materials_dir.iterdir() 
#         if d.is_dir() and not d.name.startswith('.')
#     ])
    
#     print(f"Found materials: {materials}")  # Debug output
#     return materials

# # Update materials dropdown initialization
# material_dropdown.options = find_materials()

# def find_nc_files(material):
#     return sorted((parsed_datadir / material).rglob('*.nc'))

# # Widget update handlers
# def update_files(*args):
#     if material_dropdown.value:
#         nc_files = find_nc_files(material_dropdown.value)
#         file_dropdown.options = [f.relative_to(parsed_datadir/material_dropdown.value) for f in nc_files]

# material_dropdown.observe(update_files, 'value')

# # Statistics calculation
# def my_statistics(data_array):
#     spatial_dims = [d for d in data_array.dims if d.startswith('X_')]
#     data_array = data_array.stack(spectra=spatial_dims) if 'spectra' not in data_array.dims else data_array
    
#     exp_val = data_array.mean(dim='spectra')
#     variance = data_array.var(dim='spectra')
#     std = data_array.std(dim='spectra')
#     z1 = (data_array - exp_val)/std
    
#     return {
#         'exp_val': exp_val,
#         'variance': variance,
#         'std': std,
#         'skewness': (z1**3).mean(dim='spectra'),
#         'kurtosis': (z1**4).mean(dim='spectra'),
#         'scale_param': variance / exp_val,
#         'shape_param': (exp_val**2) / (std**2)
#     }

# # Main plotting function with preprocessing
# def update_plots(material, file_path):
#     with output:
#         clear_output(wait=True)
#         plt.close('all')
        
#         # Load and preprocess
#         full_path = parsed_datadir / material / file_path
#         ds = xr.open_dataset(full_path)
#         da = ds.to_array().squeeze().rename('intensity')
#         processed_da = preprocess_dataset(da, sup_norm_threshold=0.15, option='proper_bg_proper_norm_with_outliers')
        
#         # Validate dimensions
#         if 'wave_number' not in processed_da.coords:
#             raise ValueError("Missing wave_number coordinate after preprocessing")
            
#         wave_number = processed_da.wave_number.values
#         spatial_dims = [d for d in processed_da.dims if d.startswith('X_')]
        
#         # Force correct dimension order after preprocessing
#         processed_da = processed_da.transpose(..., 'wave_number')
        
#         # Stack and ensure final dimensions
#         stacked_da = processed_da.stack(spectra=spatial_dims)
#         stacked_da = stacked_da.transpose('spectra', 'wave_number')  # Critical fix
#         spectra = stacked_da.values
        
#         # Savgol filter
#         for i in range(1000):
#             spectra = savgol_filter(spectra, window_length=7, polyorder=3, axis=0)
        
#         # Plotting code (verified correct)
#         fig_spectra = plt.figure(num='Spectra', figsize=(20, 10))
#         ax = fig_spectra.add_subplot(111)
        
#         # Plot individual spectra (correct orientation)
#         for i in range(spectra.shape[0]):  # 570 spectra
#             ax.plot(wave_number, spectra[i, :], alpha=0.5, linewidth=0.5)
        
#         # Plot mean/median (correct axis)
#         # ax.plot(wave_number, spectra.mean(axis=0), color='red', label='Mean')
#         ax.plot(wave_number, np.median(spectra, axis=0), color='blue', label='Median')
#         # ax.plot(wave_number, (1+0.15)*np.median(spectra, axis=0), color='green', linestyle='--', linewidth=3, label='Median + 15%')
#         # ax.plot(wave_number, (1-0.15)*np.median(spectra, axis=0), color='green', linestyle='--', linewidth=3, label='Median - 15%')
        

        
#         # Configure plot
#         ax.set_title(f'{material} - {file_path.name}', fontsize='x-large')
#         ax.set_xlabel('Wave Number', fontsize='x-large')
#         ax.set_ylabel('Intensity', fontsize='x-large')
#         ax.tick_params(labelsize='x-large')
#         ax.legend(prop={'size': 'x-large'})
#         ax.grid(True, alpha=0.3)
        
#         fig_median = plt.figure(num='Median', figsize=(20, 10))
#         ax = fig_median.add_subplot(111)
#         for i in range(spectra.shape[0]):  # 570 spectra
#             ax.plot(wave_number, abs(spectra[i, :]-np.median(spectra, axis=0)), alpha=0.5, linewidth=0.5)
#         ax.axhline(y=0.15, color='green', linestyle='--', linewidth=3, label='Threshold')
#         ax.set_title(f'{material} - {file_path.name} - Median subtraction algorithm', fontsize='x-large')
#         ax.set_xlabel('Wave Number', fontsize='x-large')
#         ax.set_ylabel('Intensity', fontsize='x-large')
#         ax.tick_params(labelsize='x-large')
#         ax.legend(prop={'size': 'x-large'})
#         ax.grid(True, alpha=0.3)
        
#         # Statistics figure
#         stats = my_statistics(stacked_da.unstack())  # Pass the properly shaped DataArray
#         fig_stats = plt.figure(num='Statistics', figsize=(20, 30))
        
#         # Create subplots grid
#         for idx, (var, title) in enumerate([
#             ('variance', 'Variance'),
#             ('std', 'Standard Deviation'),
#             ('skewness', 'Skewness'),
#             ('kurtosis', 'Kurtosis'),
#             ('scale_param', 'Scale Parameter'),
#             ('shape_param', 'Shape Parameter')
#         ]):
#             ax = fig_stats.add_subplot(3, 2, idx+1)
#             ax.scatter(stats['exp_val'], stats[var], alpha=0.6, s=50)
#             ax.set_title(title, fontsize='x-large')
#             ax.set_xlabel('Mean Value', fontsize='x-large')
#             ax.set_ylabel(title, fontsize='x-large')
#             ax.tick_params(labelsize='x-large')
#             ax.grid(True, alpha=0.3)

#         plt.show()

# # Connect and display widgets
# widgets.interactive_output(update_plots, {'material': material_dropdown, 'file_path': file_dropdown})
# display(widgets.VBox([material_dropdown, file_dropdown, output]))